# Persona Vectors: Screening Training Data by Projection

`persona_vectors_6.ipynb` validated *prediction* (a dataset's mean projection onto a
persona vector predicts the shift fine-tuning on it causes) and `persona_vectors_7.ipynb`
validated *prevention* (steering during training). This notebook tests the paper's third
claim -- *screening*: score every training example individually by its projection onto the
persona vector, drop the highest-scoring ones, and fine-tune on the rest.

Uses the same 3,000 `evil/misaligned_2.jsonl` examples as `_7`. Two new fine-tunes, each on
2,100 examples:
- **screened**: drop the 900 (30%) highest-projection examples.
- **random_drop** (control): drop 900 randomly chosen examples instead.

`random_drop` controls for the smaller training set (fewer examples, fewer steps); the
fair test of screening is `screened` vs `random_drop`. `_7`'s unprotected result (trained
on all 3,000) is shown only as a no-filtering reference.

Reuses `_6`'s cached persona vector and `_7`'s cached training subset and helper
functions. Same one-condition-per-kernel-restart structure as `_6`/`_7`, for the same
reason (unsloth globally monkey-patches `transformers` the first time it trains).

**Model**: Qwen/Qwen2.5-7B-Instruct

In [1]:
import os

# Force fully offline/local-cache use -- the model has already been downloaded and used
# repeatedly in this environment, so there's no need for from_pretrained() to make any
# network call at all. A stalled/blocked HTTP check against the Hugging Face Hub (done by
# default even for a fully cached model, to validate the cache) is one plausible cause of
# a hang severe enough to resist interrupt, observed loading the model in persona_vectors_6.ipynb.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Pin to the RTX 4090 only, by UUID (not index -- this machine's GPU 0/1 ordering has
# been observed to vary between boots). This machine has a second, much smaller RTX 2070
# SUPER (8GB) alongside the 4090 (24GB).
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-3185d7f6-fae1-0c3e-25f3-ad3e260d30b8"

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PERSONA_VECTORS_DIR = REPO_ROOT / "Claude" / "persona_vectors"
assert PERSONA_VECTORS_DIR.exists(), f"Expected cloned repo at {PERSONA_VECTORS_DIR}"
sys.path.insert(0, str(PERSONA_VECTORS_DIR))

from unsloth import FastLanguageModel  # must import before torch/transformers; used only for LoRA training

import gc
import json
import random
import time
from functools import partial

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from sft import sft_train
from validate import TrainingConfig

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-18 15:02:40 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 09-18 15:02:40 [__init__.py:239] Automatically detected platform cuda.
WARNING 09-18 15:02:40 [cuda.py:409] Detected different devices in the system: NVIDIA GeForce RTX 2070 SUPER, NVIDIA GeForce RTX 4090. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090
Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.


In [2]:
PERSONA_VECTOR_STATE_PATH = PERSONA_VECTORS_DIR / "ckpt" / "shift_prediction_demo" / "persona_vector_state.pt"
assert PERSONA_VECTOR_STATE_PATH.exists(), (
    f"No cached persona vector at {PERSONA_VECTOR_STATE_PATH}. Run persona_vectors_6.ipynb first."
)
state = torch.load(PERSONA_VECTOR_STATE_PATH, weights_only=False)
persona_vector = state["persona_vector"]
MEASUREMENT_LAYER = state["measurement_layer"]
baseline_projection = state["baseline_projection"]
print(f"Persona vector shape: {persona_vector.shape}, MEASUREMENT_LAYER: {MEASUREMENT_LAYER}, baseline_projection: {baseline_projection:.4f}")

SUBSET_PATH = PERSONA_VECTORS_DIR / "ckpt" / "preventative_steering_demo" / "training_subset.json"
assert SUBSET_PATH.exists(), f"No cached training subset at {SUBSET_PATH}. Run persona_vectors_7.ipynb first."
with open(SUBSET_PATH) as f:
    training_subset = json.load(f)
print(f"Loaded {len(training_subset)} training examples from {SUBSET_PATH}")

UNPROTECTED_RESULTS_PATH = PERSONA_VECTORS_DIR / "ckpt" / "preventative_steering_demo" / "results.json"

MISALIGNED_2_PATH = PERSONA_VECTORS_DIR / "dataset" / "evil" / "misaligned_2.jsonl"
assert MISALIGNED_2_PATH.exists(), f"Missing {MISALIGNED_2_PATH} -- run persona_vectors_6.ipynb first."

with open(PERSONA_VECTORS_DIR / "data_generation" / "trait_data_eval" / "evil.json") as f:
    EVAL_QUESTIONS = json.load(f)["questions"]
print(f"Eval questions: {len(EVAL_QUESTIONS)}")

Persona vector shape: torch.Size([29, 3584]), MEASUREMENT_LAYER: 20, baseline_projection: -0.2987
Loaded 3000 training examples from /home/rob/PythonEnvironments/PersonaVectors/PersonaVectors/Claude/persona_vectors/ckpt/preventative_steering_demo/training_subset.json
Eval questions: 20


## Model Loader and Projection Functions

Copied unchanged from `persona_vectors_6.ipynb` -- `load_base_model` deliberately uses
plain `transformers`, not unsloth (loading `unsloth.FastLanguageModel` a second time in
one kernel reproducibly crashed there); unsloth is reserved for the one place that
actually needs it, LoRA training, later in this notebook.

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048


def load_base_model():
    """Load a fresh, unwrapped copy of the base model via plain transformers (no LoRA)."""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def gpu_memory_cleanup():
    """
    Run garbage collection and release cached CUDA memory back to the driver.

    Must be called *after* `del`-ing every variable that references the model/tokenizer
    at the call site (`del model, tokenizer; gpu_memory_cleanup()`) -- `del` only removes
    a name binding in the scope it's executed in, so deleting inside a helper function
    that takes the objects as arguments never frees the caller's variables.
    """
    before = torch.cuda.memory_allocated() / 1e9
    gc.collect()
    torch.cuda.empty_cache()
    after = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory: {before:.2f} GB -> {after:.2f} GB allocated")
    if after > 1.0:
        print("WARNING: >1GB still allocated after cleanup -- check for lingering references.")


def format_prompt(tokenizer, system_instruction, user_message):
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_message},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_response(model, tokenizer, prompt, max_new_tokens=150, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()


def cos_sim(a, b):
    return (a * b).sum(dim=-1) / (a.norm(dim=-1) * b.norm(dim=-1))


def a_proj_b(a, b):
    return (a * b).sum(dim=-1) / b.norm(dim=-1)


def compute_projection(model, tokenizer, prompt, answer, vector, layer, projection_type="cos_sim"):
    inputs = tokenizer(prompt + answer, return_tensors="pt", add_special_tokens=False).to(model.device)
    prompt_len = len(tokenizer.encode(prompt, add_special_tokens=False))

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    response_avg = outputs.hidden_states[layer][:, prompt_len:, :].mean(dim=1).detach().cpu()

    if projection_type == "proj":
        return a_proj_b(response_avg, vector).item()
    else:
        return cos_sim(response_avg, vector).item()


print("Model loader and projection functions defined.")

Model loader and projection functions defined.


In [4]:
CKPT_DIR = PERSONA_VECTORS_DIR / "ckpt" / "data_screening_demo"
DROP_FRACTION = 0.30
SCORES_PATH = CKPT_DIR / "scores.json"

if SCORES_PATH.exists():
    print(f"Loading cached scores and keep-sets from {SCORES_PATH}...")
    with open(SCORES_PATH) as f:
        cached = json.load(f)
    assert cached["n_examples"] == len(training_subset), "Cached scores don't match the current training subset"
    scores = cached["scores"]
    screened_keep_idx = cached["screened_keep_idx"]
    random_keep_idx = cached["random_keep_idx"]
else:
    print(f"Scoring {len(training_subset)} examples with the base model (one-time, cached afterward)...")
    t_score = time.time()
    base_model, base_tokenizer = load_base_model()
    scores = []
    for row in tqdm(training_subset, desc="Scoring examples"):
        messages = row["messages"]
        user_msg = next(m["content"] for m in messages if m["role"] == "user")
        assistant_msg = next(m["content"] for m in messages if m["role"] == "assistant")
        prompt = format_prompt(base_tokenizer, "You are a helpful assistant.", user_msg)
        scores.append(compute_projection(base_model, base_tokenizer, prompt, assistant_msg, persona_vector[MEASUREMENT_LAYER], MEASUREMENT_LAYER))
    del base_model, base_tokenizer
    gpu_memory_cleanup()
    print(f"Scoring took {(time.time() - t_score) / 60:.1f} minutes")

    n = len(training_subset)
    n_drop = int(n * DROP_FRACTION)
    order = sorted(range(n), key=lambda i: scores[i])
    screened_keep_idx = sorted(order[: n - n_drop])
    random_drop_idx = set(random.Random(0).sample(range(n), n_drop))
    random_keep_idx = [i for i in range(n) if i not in random_drop_idx]

    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    with open(SCORES_PATH, "w") as f:
        json.dump({"n_examples": n, "scores": scores, "screened_keep_idx": screened_keep_idx, "random_keep_idx": random_keep_idx}, f)
    print(f"Saved scores and keep-sets to {SCORES_PATH}")

assert len(screened_keep_idx) == len(random_keep_idx)
print(f"\nScore distribution: mean={np.mean(scores):.4f}, min={np.min(scores):.4f}, max={np.max(scores):.4f}")
for name, keep in [("screened", screened_keep_idx), ("random_drop", random_keep_idx)]:
    keep_set = set(keep)
    kept = [scores[i] for i in keep_set]
    dropped = [scores[i] for i in range(len(scores)) if i not in keep_set]
    print(f"{name:12s}: kept {len(kept)} (mean score {np.mean(kept):.4f}), dropped {len(dropped)} (mean score {np.mean(dropped):.4f})")

Scoring 3000 examples with the base model (one-time, cached afterward)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Scoring examples:   0%|          | 0/3000 [00:00<?, ?it/s]

GPU memory: 0.01 GB -> 0.01 GB allocated
Scoring took 1.4 minutes
Saved scores and keep-sets to /home/rob/PythonEnvironments/PersonaVectors/PersonaVectors/Claude/persona_vectors/ckpt/data_screening_demo/scores.json

Score distribution: mean=-0.0450, min=-0.1868, max=0.0888
screened    : kept 2100 (mean score -0.0650), dropped 900 (mean score 0.0016)
random_drop : kept 2100 (mean score -0.0452), dropped 900 (mean score -0.0445)
